# CNN Classifier and Conformal Calibration

**SENTINEL-CXR** — Uncertainty-Aware Chest Radiograph Triage
Deep Learning (MAIB AI 114) · Prof Anshul Gupta · S P Jain School of Global Management, Dubai

| Group member | Student ID |
|---|---|
| Krishna Mathur | AS25DXB018 |
| Atharva Soundankar | AS25DXB020 |
| Yash Petkar | AS25DXB021 |

---

**Syllabus mapping — Week 3: Convolutional Neural Networks**

The production model. A DenseNet-121 is fine-tuned for 14-label multi-label
classification, then a **split conformal calibrator** is fitted on a held-out,
patient-disjoint calibration split.

This notebook produces `conformal_calibration.json`, the artefact the deployed
API loads. Until it is generated and deployed, the running system reports that
its coverage guarantee is *not* in force — it never pretends otherwise.

Two decisions carry most of the methodological weight:

1. **Patient-disjoint splitting.** See `patient_disjoint_split` above.
2. **AUROC, not accuracy.** With ~1% positive rate for `Hernia`, a model that
   predicts "absent" always scores 99% accuracy and is worthless.


In [ ]:
# ── Environment ───────────────────────────────────────────────────────
# Runs on Colab free tier (T4). Nothing here needs a paid runtime.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "-q", "install",
         "torchxrayvision", "scikit-learn", "seaborn"],
        check=False,
    )

import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 20260812
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | device {DEVICE}")

plt.rcParams.update({
    "figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 9, "axes.grid": True, "grid.alpha": 0.25,
})
INSTRUMENT, STAT = "#2E9CB8", "#D64541"

In [ ]:
# ── Data ──────────────────────────────────────────────────────────────
# NIH ChestX-ray14: 112,120 frontal radiographs, 30,805 patients, 14 labels.
# Kaggle: https://www.kaggle.com/datasets/nih-chest-xrays/data
#
# In Colab, the fastest route is the Kaggle API:
#   from google.colab import files; files.upload()      # kaggle.json
#   !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
#   !kaggle datasets download -d nih-chest-xrays/data -p /content/nih --unzip

DATA_DIR = os.environ.get("NIH_DIR", "/content/nih")
META = os.path.join(DATA_DIR, "Data_Entry_2017.csv")

PATHOLOGIES = ["Atelectasis","Cardiomegaly","Consolidation","Edema","Effusion",
               "Emphysema","Fibrosis","Hernia","Infiltration","Mass","Nodule",
               "Pleural_Thickening","Pneumonia","Pneumothorax"]

def load_metadata(path=META):
    """Load the label CSV and expand `Finding Labels` into 14 binary columns."""
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    for p in PATHOLOGIES:
        df[p] = df["Finding Labels"].str.contains(p, regex=False).astype(int)
    df["Patient Age"] = pd.to_numeric(df["Patient Age"], errors="coerce")
    # Ages above ~100 in this dataset are data-entry errors, not centenarians.
    df = df[(df["Patient Age"] > 0) & (df["Patient Age"] < 100)]
    return df

def patient_disjoint_split(df, fracs=(0.70, 0.10, 0.20), seed=SEED):
    """Split by Patient ID — NEVER by image.

    A patient contributes 3-4 follow-up studies. Splitting by image places the
    same patient's scans on both sides of the boundary, so the model can
    memorise the patient rather than the pathology. Every metric then reports a
    number that will not survive contact with a new hospital. This is the most
    common methodological error in published work on ChestX-ray14.
    """
    patients = df["Patient ID"].unique()
    rng = np.random.default_rng(seed)
    rng.shuffle(patients)
    n = len(patients)
    a, b = int(fracs[0] * n), int((fracs[0] + fracs[1]) * n)
    sets = (set(patients[:a]), set(patients[a:b]), set(patients[b:]))
    train, cal, test = (df[df["Patient ID"].isin(s)].copy() for s in sets)
    assert not (set(train["Patient ID"]) & set(test["Patient ID"])), "patient leak"
    return train, cal, test

## 1. Dataset and transforms

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T

class ChestXrayDataset(Dataset):
    def __init__(self, df, image_dir, train=False, size=224):
        self.df, self.image_dir = df.reset_index(drop=True), image_dir
        # Augmentation is deliberately conservative. Aggressive flips would be
        # wrong here: situs inversus is rare, so a horizontally flipped
        # radiograph teaches the model anatomy that almost never occurs.
        self.tf = T.Compose(
            ([T.RandomRotation(7), T.RandomResizedCrop(size, scale=(0.9, 1.0))]
             if train else [T.Resize((size, size))])
            + [T.ToTensor(), T.Normalize([0.485], [0.229])]
        )

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(os.path.join(self.image_dir, row["Image Index"])).convert("L")
        y = torch.tensor([row[p] for p in PATHOLOGIES], dtype=torch.float32)
        return self.tf(img), y

print("Dataset defined. Point IMAGE_DIR at the folder holding the PNGs.")

## 2. Model — DenseNet-121 with dropout

Dropout is added deliberately: without it, Monte-Carlo dropout at inference produces T identical passes and an epistemic uncertainty of exactly zero, which would be a false claim of certainty.

In [ ]:
import torchvision.models as tvm

def build_densenet(n_classes=14, dropout=0.2, pretrained=True):
    m = tvm.densenet121(weights="IMAGENET1K_V1" if pretrained else None)
    # Radiographs are single-channel. Sum the pretrained RGB filters rather
    # than discarding two of them — this preserves the learned edge detectors.
    w = m.features.conv0.weight.data.sum(dim=1, keepdim=True)
    m.features.conv0 = nn.Conv2d(1, 64, 7, 2, 3, bias=False)
    m.features.conv0.weight.data = w
    m.classifier = nn.Sequential(
        nn.Dropout(dropout),           # required for MC-dropout at inference
        nn.Linear(m.classifier.in_features, n_classes),
    )
    return m

model = build_densenet().to(DEVICE)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

## 3. Training

Positive weighting matters: without it the loss is dominated by the negative class and the model learns to predict 'absent' for every rare finding.

In [ ]:
def positive_weights(df):
    """pos_weight = (#negatives / #positives) per label, capped.

    Uncapped, `Hernia` gets a weight near 500 and its gradient drowns out the
    other thirteen labels.
    """
    w = []
    for p in PATHOLOGIES:
        pos = max(int(df[p].sum()), 1)
        w.append(min((len(df) - pos) / pos, 20.0))
    return torch.tensor(w, dtype=torch.float32)

def train_epoch(model, loader, opt, criterion, scaler=None):
    model.train(); total = 0.0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        if scaler:                       # mixed precision — ~2x faster on T4
            with torch.autocast("cuda", dtype=torch.float16):
                loss = criterion(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        else:
            loss = criterion(model(x), y); loss.backward(); opt.step()
        total += loss.item() * x.size(0)
    return total / len(loader.dataset)

@torch.no_grad()
def predict(model, loader):
    model.eval(); P, Y = [], []
    for x, y in loader:
        P.append(torch.sigmoid(model(x.to(DEVICE))).cpu().numpy()); Y.append(y.numpy())
    return np.concatenate(P), np.concatenate(Y)

print("Training utilities ready.")
print("Run: python -c 'see cell below' after pointing IMAGE_DIR at the data.")

## 4. Evaluation — AUROC per pathology

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

def evaluate(probs, labels):
    rows = []
    for i, p in enumerate(PATHOLOGIES):
        y = labels[:, i]
        if y.sum() < 5 or y.sum() == len(y):
            rows.append((p, np.nan, np.nan, int(y.sum()))); continue
        rows.append((p,
                     roc_auc_score(y, probs[:, i]),
                     average_precision_score(y, probs[:, i]),
                     int(y.sum())))
    df = pd.DataFrame(rows, columns=["pathology", "AUROC", "AUPRC", "n_positive"])
    print(df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    print(f"\nMacro AUROC: {df['AUROC'].mean():.4f}")
    print("AUPRC is reported alongside AUROC because AUROC flatters a model on")
    print("heavily imbalanced labels — it is insensitive to the false-positive")
    print("rate when negatives vastly outnumber positives.")
    return df

print("evaluate() ready")

## 5. Fit and export the conformal calibrator

This produces the artefact the deployed API loads.

In [ ]:
import json, math

def conformal_quantile(scores, alpha):
    """Finite-sample-corrected quantile. Omitting (n+1)/n is the classic bug."""
    scores = np.asarray(scores, float); n = scores.size
    if n == 0: return 1.0
    rank = math.ceil((n + 1) * (1 - alpha))
    return 1.0 if rank > n else float(np.sort(scores)[rank - 1])

def fit_conformal(cal_probs, cal_labels, alpha=0.10, min_pos=20):
    thresholds, counts = np.full(14, 0.5), np.zeros(14, int)
    for k in range(14):
        pos = cal_probs[cal_labels[:, k].astype(bool), k]
        counts[k] = pos.size
        thresholds[k] = conformal_quantile(1 - pos, alpha) if pos.size >= min_pos else 0.5
    return thresholds, counts

def empirical_coverage(probs, labels, thresholds):
    inc = (1 - probs) <= thresholds
    out = {}
    for k, name in enumerate(PATHOLOGIES):
        m = labels[:, k].astype(bool)
        out[name] = float(inc[m, k].mean()) if m.sum() else np.nan
    return out

def export_calibration(thresholds, counts, alpha=0.10, path="conformal_calibration.json"):
    json.dump({"alpha": alpha, "max_set_size": 6,
               "thresholds": thresholds.tolist(),
               "n_calibration": counts.tolist(),
               "pathologies": PATHOLOGIES}, open(path, "w"), indent=2)
    print(f"Wrote {path}")
    print("Copy it to apps/api/artifacts/ and redeploy — the API will then")
    print("report `fitted: true` and its coverage guarantee becomes real.")

print("Conformal utilities ready.")
print("\nThe validation that matters: empirical coverage on the TEST split")
print("should be >= 1 - alpha. That single number is the project's core claim.")

---

### References for this notebook

- Wang, X. et al. (2017). ChestX-ray8: Hospital-scale chest X-ray database. *CVPR*.
- Rajpurkar, P. et al. (2017). CheXNet: Radiologist-level pneumonia detection. arXiv:1711.05225.
- Angelopoulos, A. & Bates, S. (2023). Conformal prediction: a gentle introduction. *FnT ML*.
- Krizhevsky, A., Sutskever, I. & Hinton, G. (2017). ImageNet classification with deep CNNs. *CACM*.

---

*SENTINEL-CXR is a student research prototype. It is not a medical device and
must not be used for clinical decisions.*
